### Example 2:
- Weather in Bishkek? 
- Convert 100 USD to KGS.
- What is 25 + 17?

The new problem is knowing which function to run.
Use a dictionary called registry

In [77]:
from openai import OpenAI, pydantic_function_tool
from IPython.display import display, Markdown
from pydantic import BaseModel, Field

In [78]:
MODEL_ID = "llama3.1:8b"

RATES_TO_USD: dict[str, float] = {
    "USD": 1.0,
    "KGS": 86.0,
    "EUR": 0.92,
}

client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1",
)


In [79]:
class GetWeather(BaseModel):
    """Get the current weather for one city"""
    city: str = Field(definition="Name of the city, e.g. Bishkek")
    
class ConvertCurrency(BaseModel):
    """Convert the currency for given source currency to destination currency, e.g. USD to KGS, or vise-versa """
    amount: float = Field(description="Amount to be converted, e.g. 100USD")
    from_currency: str = Field(description="Shortcut for the source currency, e.g. USD")
    to_currency: str = Field(description="Shortcut for the destination currency, e.g. USD")
    
class Add(BaseModel):
    """Add two given numbers."""
    x: float = Field(description="First number, e.g. 4")
    y: float = Field(description="First number, e.g. 4")

/var/folders/x4/25yf81sn1kqb18j0dmflttb00000gp/T/ipykernel_63599/1770361990.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'definition'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  city: str = Field(definition="Name of the city, e.g. Bishkek")


In [87]:
def get_weather(args: GetWeather) -> dict:
    fake = {
        "Bishkek": "16C", "Naryn": "10C", "Osh": "30C", "Tokmok": "25C"
    }
    return {args.city: fake[args.city]}

def convert_currency(args: ConvertCurrency) -> str:
    amount = args.amount
    from_curr = args.from_currency.upper()
    to_curr = args.to_currency.upper()
    rates: dict[str, float] = RATES_TO_USD
    
    if from_curr not in rates or to_curr not in rates:
        raise ValueError(f"Unsupported currency: {from_curr} or {to_curr}")
    
    amount_in_usd = amount / rates[from_curr]
    converted_amount = amount_in_usd * rates[to_curr]
    
    return f"{amount} {from_curr} is {round(converted_amount, 2)} {to_curr}"

def add(args: Add) -> float:
    x1 = args.x
    y1 = args.y
    
    return x1 + y1

In [81]:
# city1 = GetWeather(city="Naryn")
# print(get_weather(city1))

# curr1 = ConvertCurrency(amount=5000, from_currency="USD", to_currency="EUR")
# print(convert_currency(curr1))

# add1 = Add(x=6, y=10)
# print(add(add1))
# print(type(add(add1)))

The new problem is which function to run

In [82]:
REGISTRY = {
    "GetWeather": (GetWeather, get_weather),
    "ConverCurrency": (ConvertCurrency, convert_currency),
    "Add": (Add, add)
}
tools = [pydantic_function_tool(model) for model, _ in REGISTRY.values()]

In [83]:
for key, value in REGISTRY.values():
    print(key, value)


<class '__main__.GetWeather'> <function get_weather at 0x10d93a520>
<class '__main__.ConvertCurrency'> <function convert_currency at 0x10d939b20>
<class '__main__.Add'> <function add at 0x10d93ae80>


In [84]:
tools

[{'type': 'function',
  'function': {'name': 'GetWeather',
   'strict': True,
   'parameters': {'description': 'Get the current weather for one city',
    'properties': {'city': {'definition': 'Name of the city, e.g. Bishkek',
      'title': 'City',
      'type': 'string'}},
    'required': ['city'],
    'title': 'GetWeather',
    'type': 'object',
    'additionalProperties': False},
   'description': 'Get the current weather for one city'}},
 {'type': 'function',
  'function': {'name': 'ConvertCurrency',
   'strict': True,
   'parameters': {'description': 'Convert the currency for given source currency to destination currency, e.g. USD to KGS, or vise-versa ',
    'properties': {'amount': {'description': 'Amount to be converted, e.g. 100USD',
      'title': 'Amount',
      'type': 'number'},
     'from_currency': {'description': 'Shortcut for the source currency, e.g. USD',
      'title': 'From Currency',
      'type': 'string'},
     'to_currency': {'description': 'Shortcut for the d

In [85]:
user_prompt = """
Weather in Paris? Convert 100 USD to EUR. What is 25 + 17?"
"""
messages = [{"role": "user", "content": user_prompt}]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    tools=tools,
    reasoning_effort="none"
)
assistant = response.choices[0].message

In [ ]:
for tool_call in assistant.tool_calls:
    model, function = REGISTRY[tool_call.function.name]
    args = model.model_validate_json(tool_call.function.arguments)
    result = function(args)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result
    })
final = client.chat.completions.create(
    model="gemma4:e4b",
    messages=messages,
    reasoning_effort="none"
)

KeyError: 'Paris'